# Requirement
## 1. Load data from flight-time.json into a table
## 2. Table structure is given below

  FL_DATE DATE, 

  OP_CARRIER STRING, 

  OP_CARRIER_FL_NUM STRING, 

  ORIGIN STRING, 

  ORIGIN_CITY_NAME STRING, 

  DEST STRING, 

  DEST_CITY_NAME STRING, 

  CRS_DEP_TIME LONG, 

  DEP_TIME LONG, 

  WHEELS_ON INT, 

  TAXI_IN INT, 

  CRS_ARR_TIME LONG, 

  ARR_TIME LONG, 

  CANCELLED INT, 
  
 DISTANCE INT

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, LongType

schema= StructType([
                    StructField('FL_DATE', DateType()),
                    StructField('OP_CARRIER', StringType()),
                    StructField('OP_CARRIER_FL_NUM',StringType()), 
                    StructField('ORIGIN', StringType()),
                    StructField('ORIGIN_CITY_NAME', StringType()),
                    StructField('DEST', StringType()),
                    StructField('DEST_CITY_NAME', StringType()),
                    StructField('CRS_DEP_TIME', LongType()),
                    StructField('DEP_TIME', LongType()),
                    StructField('WHEELS_ON', IntegerType()),
                    StructField('TAXI_IN', IntegerType()),
                    StructField('CRS_ARR_TIME', LongType()),
                    StructField('ARR_TIME', LongType()),
                    StructField('CANCELLED', IntegerType()),
                    StructField('DISTANCE', IntegerType())
])

reading_df=(
                spark.read
                        .format('json')
                        .option('mode', 'FAILFAST')
                        .schema(schema)
                        .option('dateFormat','M/d/yyyy')
                        .load('/Volumes/dev/spark_db/datasets/spark_programming/data/flight-time.json')
)

reading_df.display()

In [0]:
(
    reading_df.write
            .mode('overwrite')
            .saveAsTable('dev.spark_db.flight_time_raw')
)

## Requirement
### 1. Read raw data from flight_time_raw table
### 2. Apply transformations to time values as hour to minute interval

     1. CRS_DEP_TIME
     2. DEP_TIME
     3. WHEELS_ON
     4. CRS_ARR_TIME
     5. ARR_TIME
     
### 3. Apply transformation to TAXI_IN to make it a minute interval

1. Read raw data from flight_time_raw table

In [0]:
raw_df=spark.sql('SELECT * FROM dev.spark_db.flight_time_raw')
raw_df.display()

2. Apply transformations to time values as hour to minute interval
    1. CRS_DEP_TIME
    2. DEP_TIME
    3. WHEELS_ON
    4. CRS_ARR_TIME
    5. ARR_TIME

In [0]:
from pyspark.sql import functions as F
trans_df1=(
        raw_df.withColumns({
            'CRS_DEP_TIME' : F.expr("CAST(CONCAT((LEFT(LPAD(CRS_DEP_TIME,4,'0'),2)), ':',(RIGHT(LPAD(CRS_DEP_TIME,4,'0'),2))) AS INTERVAL HOUR TO MINUTE)"),
            'DEP_TIME' : F.expr("CAST(CONCAT((LEFT(LPAD(DEP_TIME,4,'0'),2)), ':', (RIGHT(LPAD(DEP_TIME,4,'0'),2))) AS INTERVAL HOUR TO MINUTE)"),
            'WHEELS_ON' : F.expr("CAST(CONCAT((LEFT(LPAD(WHEELS_ON,4,'0'),2)), ':', (RIGHT(LPAD(WHEELS_ON,4,'0'),2))) AS INTERVAL HOUR TO MINUTE)"),
            'CRS_ARR_TIME' : F.expr("CAST(CONCAT((LEFT(LPAD(CRS_ARR_TIME,4,'0'),2)), ':', (RIGHT(LPAD(CRS_ARR_TIME,4,'0'),2))) AS INTERVAL HOUR TO MINUTE)"), 
            'ARR_TIME' : F.expr("CAST(CONCAT((LEFT(LPAD(ARR_TIME,4,'0'),2)), ':', (RIGHT(LPAD(ARR_TIME,4,'0'),2))) AS INTERVAL HOUR TO MINUTE)")         
        })
)


trans_df1.display()

3. Apply transformation to TAXI_IN to make it a minute interval

In [0]:

trans_df2=trans_df1.withColumn('TAXI_IN',F.col('TAXI_IN').cast('interval minute'))

(
    trans_df2.write
            .format('delta')
            .mode('overwrite')
            .saveAsTable("dev.spark_db.flight_time")
)